# Notebook — Train Type-Aware Compatibility Embedding (ResNet-18 + Triplet Loss)

Trains a ResNet-18 backbone with type-specific masking on the Polyvore Outfits dataset.
Uses triplet loss with category-constrained hard negatives so the embedding space
encodes outfit compatibility, not just visual similarity.

**Input (add as Kaggle dataset inputs):**
- `polyvore-outfits` dataset → mounted at `/kaggle/input/polyvore-outfits/`

**Output:**
- `compatibility_embedding.onnx` — embedding network (input: 224×224 image, output: 64-dim L2-normalised vector)
- `compatibility_embedding_int8.onnx` — INT8 quantised version for faster CPU inference

**What makes this different from FashionCLIP:**
- ResNet-18 is fine-tuned end-to-end with **triplet loss on outfit pairs** — the embedding space is
  explicitly trained so compatible items cluster together, even if they look visually different.
- FashionCLIP was trained for text-image retrieval — compatible items are NOT guaranteed to be close.

**Run on:** Kaggle T4 GPU (~2-3 hours)

In [ ]:
# 1. Install dependencies
!pip install -q torch torchvision onnx onnxruntime onnxscript scikit-learn matplotlib tqdm Pillow numpy

In [ ]:
# 2. Imports and reproducibility
import os, json, pickle, random
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
from tqdm import tqdm

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
# 3. Config and path detection
OUTPUT_DIR = '/kaggle/working'
SPLIT      = 'nondisjoint'
DIM_EMBED  = 64
IMG_SIZE   = 224
BATCH_SIZE = 128   # reduce to 64 if OOM
LR         = 5e-5
MAX_EPOCHS = 20
PATIENCE   = 5
MARGIN     = 0.3

# Auto-detect dataset root — handles different Kaggle mount structures
_candidates = [
    '/kaggle/input/polyvore-outfits/polyvore_outfits',
    '/kaggle/input/polyvore-outfits',
]
DATA_DIR = None
for _c in _candidates:
    if Path(_c).is_dir() and (Path(_c) / 'images').is_dir():
        DATA_DIR = _c
        break

if DATA_DIR is None:
    print('Could not auto-detect dataset root. Contents of /kaggle/input:')
    for p in Path('/kaggle/input').rglob('*'):
        if p.is_dir() and p.name == 'images':
            print(f'  Found images/ at: {p.parent}')
    raise FileNotFoundError('Set DATA_DIR manually to the folder that contains images/ and polyvore_item_metadata.json')

print(f'Dataset root: {DATA_DIR}')
print(f'Images dir:   {DATA_DIR}/images')
print(f'Split dir:    {DATA_DIR}/{SPLIT}')

assert Path(f'{DATA_DIR}/images').is_dir(), 'images/ folder missing'
assert Path(f'{DATA_DIR}/{SPLIT}/train.json').is_file(), f'{SPLIT}/train.json missing'
assert Path(f'{DATA_DIR}/polyvore_item_metadata.json').is_file(), 'polyvore_item_metadata.json missing'
assert Path(f'{DATA_DIR}/{SPLIT}/typespaces.p').is_file(), f'{SPLIT}/typespaces.p missing'
print('All required files found.')

In [ ]:
# 4. Load metadata and typespaces

# Item metadata: item_id -> {semantic_category, title, ...}
print('Loading item metadata...')
with open(f'{DATA_DIR}/polyvore_item_metadata.json') as f:
    meta = json.load(f)

print(f'  Items in metadata: {len(meta):,}')

# Sample to understand structure
_sample_id = next(iter(meta))
print(f'  Sample item_id: {_sample_id}')
print(f'  Sample fields:  {list(meta[_sample_id].keys())}')
print(f'  Sample semantic_category: {meta[_sample_id].get("semantic_category", "N/A")}')

# Typespaces: list of (category_1, category_2) tuples defining type-pair embedding spaces
print('\nLoading typespaces...')
with open(f'{DATA_DIR}/{SPLIT}/typespaces.p', 'rb') as f:
    try:
        typespaces_list = pickle.load(f)
    except UnicodeDecodeError:
        f.seek(0)
        typespaces_list = pickle.load(f, encoding='latin1')

# Build lookup dict: (cat1, cat2) -> condition_index
typespaces = {t: i for i, t in enumerate(typespaces_list)}
n_conditions = len(typespaces)
print(f'  Type-pair conditions (n_conditions): {n_conditions}')
print(f'  Sample typespace entries: {typespaces_list[:3]}')

In [ ]:
# 5. Dataset class — triplet sampling with category-constrained hard negatives

class PolyvoreDataset(Dataset):
    """
    Returns (anchor, positive, negative, condition_index) triplets.
    - Positive pairs: two items from the same outfit
    - Negative: item from a DIFFERENT outfit but the SAME semantic category as the positive
      (category-constrained hard negative — harder to distinguish than random)
    - condition_index: index into the type-specific embedding space for this (anchor_cat, pos_cat) pair
    """

    def __init__(self, data_dir, split, typespaces, meta, transform, subset=None):
        self.img_dir    = Path(data_dir) / 'images'
        self.transform  = transform
        self.typespaces = typespaces

        # Build item -> semantic_category mapping
        self.im2cat = {
            str(iid): str(info.get('semantic_category', 'unknown'))
            for iid, info in meta.items()
        }

        # Load outfit data
        with open(Path(data_dir) / split / 'train.json') as f:
            outfits = json.load(f)

        if subset:
            outfits = outfits[:subset]

        # category -> [(outfit_idx, item_id)] for negative sampling
        self.cat2items = defaultdict(list)
        # Positive pairs: (outfit_idx, anchor_id, pos_id, condition, pos_cat)
        self.pos_pairs = []

        for oidx, outfit in enumerate(outfits):
            items    = outfit.get('items', [])
            item_ids = [str(item['item_id']) for item in items]

            for iid in item_ids:
                cat = self.im2cat.get(iid, 'unknown')
                self.cat2items[cat].append((oidx, iid))

            for i in range(len(item_ids)):
                for j in range(len(item_ids)):
                    if i == j:
                        continue
                    anchor_id  = item_ids[i]
                    pos_id     = item_ids[j]
                    anchor_cat = self.im2cat.get(anchor_id, 'unknown')
                    pos_cat    = self.im2cat.get(pos_id, 'unknown')
                    key        = (anchor_cat, pos_cat)
                    if key in self.typespaces:
                        condition = self.typespaces[key]
                        self.pos_pairs.append((oidx, anchor_id, pos_id, condition, pos_cat))

        print(f'  {split}: {len(outfits):,} outfits  →  {len(self.pos_pairs):,} positive pairs')

    def _load_image(self, item_id):
        path = self.img_dir / f'{item_id}.jpg'
        try:
            return self.transform(Image.open(path).convert('RGB'))
        except Exception:
            return torch.zeros(3, IMG_SIZE, IMG_SIZE)

    def _sample_negative(self, outfit_idx, pos_cat):
        candidates = self.cat2items[pos_cat]
        # Try up to 10 times to find a negative from a different outfit
        for _ in range(10):
            oidx, neg_id = random.choice(candidates)
            if oidx != outfit_idx:
                return neg_id
        # Fall back: use any item from this category
        return random.choice(candidates)[1]

    def __len__(self):
        return len(self.pos_pairs)

    def __getitem__(self, idx):
        oidx, anchor_id, pos_id, condition, pos_cat = self.pos_pairs[idx]
        neg_id = self._sample_negative(oidx, pos_cat)
        return (
            self._load_image(anchor_id),
            self._load_image(pos_id),
            self._load_image(neg_id),
            torch.tensor(condition, dtype=torch.long),
        )

In [ ]:
# 6. Model definition

class EmbeddingNet(nn.Module):
    """ResNet-18 backbone with a custom embedding head."""
    def __init__(self, dim_embed=64):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.features = nn.Sequential(*list(resnet.children())[:-1])  # drop final FC
        self.fc       = nn.Linear(512, dim_embed)

    def forward(self, x):
        x = self.features(x)   # (B, 512, 1, 1)
        x = x.flatten(1)       # (B, 512)
        return self.fc(x)      # (B, dim_embed)


class TypeAwareNet(nn.Module):
    """
    Wraps EmbeddingNet with learned type-specific diagonal masks.
    Each (anchor_category, positive_category) pair has its own mask
    that projects the general embedding into a compatibility-specific subspace.

    At INFERENCE we use the general L2-normalised embedding (no mask)
    which is still trained for compatibility via the triplet loss.
    """
    def __init__(self, dim_embed=64, n_conditions=1):
        super().__init__()
        self.embedding_net = EmbeddingNet(dim_embed)
        self.masks = nn.Embedding(n_conditions, dim_embed)

        # Initialise masks to near-identity (disjoint sections)
        mask_array = np.zeros([n_conditions, dim_embed], dtype=np.float32)
        mask_len   = max(1, dim_embed // n_conditions)
        for i in range(n_conditions):
            start = (i * mask_len) % dim_embed
            mask_array[i, start:start + mask_len] = 1.0
        self.masks.weight = nn.Parameter(torch.from_numpy(mask_array))

    def forward(self, x, condition=None):
        emb = self.embedding_net(x)            # (B, dim_embed)
        if condition is None:
            # General embedding — used at inference
            return F.normalize(emb, p=2, dim=1)
        # Type-specific embedding — used during training
        mask   = F.relu(self.masks(condition)) # (B, dim_embed)
        masked = emb * mask
        return F.normalize(masked, p=2, dim=1)


model = TypeAwareNet(dim_embed=DIM_EMBED, n_conditions=n_conditions).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {total_params:,}')
print(f'  EmbeddingNet:  {sum(p.numel() for p in model.embedding_net.parameters()):,}')
print(f'  Type masks:    {sum(p.numel() for p in model.masks.parameters()):,}')

In [ ]:
# 7. Transforms and DataLoaders

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

print('Building train dataset...')
train_ds = PolyvoreDataset(DATA_DIR, SPLIT, typespaces, meta, train_transform)
print('Building val dataset (first 10% of train outfits as proxy val)...')
val_ds   = PolyvoreDataset(DATA_DIR, SPLIT, typespaces, meta, eval_transform, subset=6000)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=4, pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=4, pin_memory=True,
)

print(f'Train batches: {len(train_loader):,}')
print(f'Val batches:   {len(val_loader):,}')

In [ ]:
# 8. Training setup

criterion = nn.TripletMarginLoss(margin=MARGIN, p=2, reduction='mean')
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=2, factor=0.5, min_lr=1e-6
)

def run_epoch(loader, training):
    model.train() if training else model.eval()
    total_loss, n_batches = 0.0, 0

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for anchor, pos, neg, condition in tqdm(loader, leave=False):
            anchor    = anchor.to(DEVICE)
            pos       = pos.to(DEVICE)
            neg       = neg.to(DEVICE)
            condition = condition.to(DEVICE)

            emb_a = model(anchor, condition)
            emb_p = model(pos,    condition)
            emb_n = model(neg,    condition)

            loss = criterion(emb_a, emb_p, emb_n)

            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            n_batches  += 1

    return total_loss / n_batches

In [ ]:
# 9. Training loop

best_val_loss = float('inf')
best_state    = None
patience_cnt  = 0
train_losses, val_losses = [], []

print('Training ...')
for epoch in range(1, MAX_EPOCHS + 1):
    train_loss = run_epoch(train_loader, training=True)
    val_loss   = run_epoch(val_loader,   training=False)
    scheduler.step(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    lr_now = optimizer.param_groups[0]['lr']
    print(f'Epoch {epoch:02d}/{MAX_EPOCHS}  '
          f'train={train_loss:.4f}  val={val_loss:.4f}  lr={lr_now:.2e}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state    = {k: v.clone() for k, v in model.state_dict().items()}
        patience_cnt  = 0
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f'Early stopping at epoch {epoch} (best val loss: {best_val_loss:.4f})')
            break

model.load_state_dict(best_state)
print(f'\nBest val loss: {best_val_loss:.4f}')

In [ ]:
# 10. Plot training curves

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(train_losses, label='Train loss')
ax.plot(val_losses,   label='Val loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Triplet loss')
ax.set_title('Training curves')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/training_curves.png', dpi=120)
plt.show()

In [ ]:
# 11. Evaluate compatibility AUC on test set
#
# Uses compatibility_test.txt — each row: label item_id_1 item_id_2 ...
# Score = average pairwise cosine similarity of general embeddings
# Higher score = more compatible (1 = compatible, 0 = incompatible)

compat_file = Path(DATA_DIR) / SPLIT / 'compatibility_test.txt'
assert compat_file.is_file(), f'Not found: {compat_file}'

model.eval()
scores, labels = [], []

print('Evaluating on compatibility_test.txt ...')
with open(compat_file) as f:
    lines = f.readlines()

for line in tqdm(lines):
    parts = line.strip().split()
    if len(parts) < 3:
        continue
    label    = int(parts[0])
    item_ids = parts[1:]

    embs = []
    for iid in item_ids:
        img_path = Path(DATA_DIR) / 'images' / f'{iid}.jpg'
        try:
            img    = Image.open(img_path).convert('RGB')
            tensor = eval_transform(img).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                emb = model(tensor)  # general L2-normalised embedding
            embs.append(emb.cpu())
        except Exception:
            continue

    if len(embs) < 2:
        continue

    embs   = torch.cat(embs, 0)  # (N, DIM_EMBED)
    # Pairwise cosine similarity (embeddings are already L2-normalised → dot product = cosine sim)
    sim_mat = embs @ embs.T      # (N, N)
    mask    = torch.triu(torch.ones(len(embs), len(embs)), diagonal=1).bool()
    avg_sim = sim_mat[mask].mean().item()

    scores.append(avg_sim)
    labels.append(label)

test_auc = roc_auc_score(labels, scores)
print(f'\nCompatibility AUC on test set: {test_auc:.4f}  (target >= 0.83)')

if test_auc < 0.75:
    print('WARNING: AUC below 0.75 — consider training longer or increasing BATCH_SIZE.')
elif test_auc >= 0.83:
    print('Target reached.')
else:
    print('Reasonable — type-specific inference would push this higher.')

In [ ]:
# 12. Export the embedding network to ONNX
#
# We export ONLY the EmbeddingNet (ResNet-18 + FC + L2 norm) — no masks needed at inference.
# The backbone is already trained to encode compatibility via the triplet loss.
# Inference: embed both items → cosine similarity → compatibility score.

import onnx
from onnxruntime.quantization import quantize_dynamic, QuantType
import onnxruntime as ort

ONNX_PATH = f'{OUTPUT_DIR}/compatibility_embedding.onnx'
INT8_PATH = f'{OUTPUT_DIR}/compatibility_embedding_int8.onnx'


class EmbeddingWrapper(nn.Module):
    """Thin wrapper: ResNet-18 backbone + FC + L2 normalisation."""
    def __init__(self, type_aware_net):
        super().__init__()
        self.net = type_aware_net.embedding_net

    def forward(self, x):
        emb = self.net(x)
        return F.normalize(emb, p=2, dim=1)  # L2 normalise


wrapper     = EmbeddingWrapper(model).eval().cpu()
dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)

with torch.no_grad():
    test_out = wrapper(dummy_input)
print(f'Wrapper output shape: {test_out.shape}')  # (1, 64)

print(f'Exporting to {ONNX_PATH} ...')
torch.onnx.export(
    wrapper,
    dummy_input,
    ONNX_PATH,
    input_names=['image'],
    output_names=['embedding'],
    dynamic_axes={'image': {0: 'batch'}, 'embedding': {0: 'batch'}},
    opset_version=17,
    do_constant_folding=True,
)

onnx.checker.check_model(ONNX_PATH)
print(f'Saved: {ONNX_PATH}  ({os.path.getsize(ONNX_PATH)/1e6:.1f} MB)')

print('Quantising to INT8 ...')
quantize_dynamic(ONNX_PATH, INT8_PATH, weight_type=QuantType.QInt8)
print(f'Saved: {INT8_PATH}  ({os.path.getsize(INT8_PATH)/1e6:.1f} MB)')

In [ ]:
# 13. Validate ONNX output matches PyTorch

sess = ort.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])

test_input = np.random.randn(1, 3, IMG_SIZE, IMG_SIZE).astype(np.float32)

with torch.no_grad():
    pt_out = wrapper(torch.from_numpy(test_input)).numpy()

onnx_out = sess.run(['embedding'], {'image': test_input})[0]

max_diff = float(np.abs(pt_out - onnx_out).max())
print(f'Max difference PyTorch vs ONNX: {max_diff:.6f}')
assert max_diff < 1e-3, f'Outputs differ too much ({max_diff})'
print('ONNX outputs match PyTorch.')

# Quick sanity check: embeddings should be L2-normalised
norm = float(np.linalg.norm(onnx_out))
print(f'Embedding L2 norm (should be ~1.0): {norm:.6f}')

In [ ]:
# 14. Sanity check: same-outfit pairs should score higher than random pairs

sess_int8 = ort.InferenceSession(INT8_PATH, providers=['CPUExecutionProvider'])

def embed_image(img_path):
    img = Image.open(img_path).convert('RGB')
    arr = eval_transform(img).unsqueeze(0).numpy()
    return sess_int8.run(['embedding'], {'image': arr})[0][0]  # (64,)

def cosine_sim(a, b):
    return float(np.dot(a, b))  # already L2-normalised

# Load a few test outfits
with open(f'{DATA_DIR}/{SPLIT}/test.json') as f:
    test_outfits = json.load(f)[:20]

print('Checking same-outfit vs cross-outfit similarity:')
same_sims, cross_sims = [], []

for i, outfit in enumerate(test_outfits[:10]):
    items = outfit.get('items', [])
    if len(items) < 2:
        continue
    ids = [str(item['item_id']) for item in items[:2]]
    try:
        e1 = embed_image(Path(DATA_DIR) / 'images' / f'{ids[0]}.jpg')
        e2 = embed_image(Path(DATA_DIR) / 'images' / f'{ids[1]}.jpg')
        same_sims.append(cosine_sim(e1, e2))
        # Cross: pair with item from a different outfit
        other_id = str(test_outfits[(i+1) % len(test_outfits)]['items'][0]['item_id'])
        e3 = embed_image(Path(DATA_DIR) / 'images' / f'{other_id}.jpg')
        cross_sims.append(cosine_sim(e1, e3))
    except Exception:
        continue

print(f'Same-outfit similarity:  mean={np.mean(same_sims):.3f}')
print(f'Cross-outfit similarity: mean={np.mean(cross_sims):.3f}')
if np.mean(same_sims) > np.mean(cross_sims):
    print('Same-outfit similarity is higher — embeddings encode compatibility correctly.')
else:
    print('WARNING: cross-outfit similarity is higher — check training or dataset paths.')

In [ ]:
# 15. Package outputs

import zipfile

zip_path = f'{OUTPUT_DIR}/type_aware_compatibility.zip'
files_to_zip = [
    (ONNX_PATH,                                   'compatibility_embedding.onnx'),
    (INT8_PATH,                                    'compatibility_embedding_int8.onnx'),
    (f'{OUTPUT_DIR}/training_curves.png',          'training_curves.png'),
]

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fpath, arcname in files_to_zip:
        if os.path.isfile(fpath):
            zf.write(fpath, arcname)
            print(f'  Added {arcname} ({os.path.getsize(fpath)/1e6:.1f} MB)')

print(f'\nZip: {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)')
print('\nNext steps:')
print('1. Kaggle sidebar > Data > Output > Save Version')
print('2. Download type_aware_compatibility.zip from the Output tab')
print('3. Extract and place in backend/model/:')
print('     compatibility_embedding_int8.onnx  (production)')
print('     compatibility_embedding.onnx       (reference)')